In [3]:
import connectorx as cx
import evadb
import ffnn
import numpy as np
import pandas as pd
import tensorflow as tf
import torch
from torch.utils.data import DataLoader
import utils
import load_data_to_db
import collections
import os
import h5py
from abc import ABC, abstractmethod
from models.preprocessing.inputs import SparseFeat, DenseFeat, VarLenSparseFeat
from models.dssm import DSSM_Torch, DSSM_TF, get_var_feature, get_test_var_feature
from sklearn.preprocessing import LabelEncoder
from tqdm.auto import tqdm
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, pandas_udf, when
import pyspark.sql.functions as F
from pyspark.sql.types import ArrayType, FloatType, StringType, IntegerType
from dssm_evadb import DSSM_Moel_Wrapper
import pickle
import multiprocessing as mp
from pipeline import Pipeline
from sklearn.model_selection import train_test_split
import pyarrow.parquet as pq


2025-01-06 06:40:24.470091: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2025-01-06 06:40:24.470118: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.1.1 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
2025-01-06 06:40:26.759591: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or d

In [130]:
# Use case 3, trainig query
query_to_fetch_training_data = """
select store, department, li_order_id, price, quantity,
EXTRACT(WEEK FROM date) AS week,
EXTRACT(MONTH FROM date) AS month,
CASE 
    WHEN EXTRACT(WEEK FROM date) > 50 AND EXTRACT(MONTH FROM date) = 1 
    THEN EXTRACT(YEAR FROM date) - 1
    ELSE EXTRACT(YEAR FROM date)
END AS year,
quantity * price as row_price
from tpcxai_order_training join tpcxai_lineitem_training on o_order_id=li_order_id
join tpcxai_product_training on li_product_id=p_product_id
"""

In [131]:
df = utils.fetch_data_from_postgres_via_psycopg2(query_to_fetch_training_data)

In [133]:
grouped = df.groupby(['store', 'department', 'year', 'week'])['row_price'].sum().reset_index()
grouped = grouped.rename(index=str, columns={'store': 'Store', 'department': 'Dept', 'date': 'Date', 'row_price': 'Weekly_Sales'})

In [134]:
grouped['num_of_week'] = (grouped['year'].astype(int) - 2010) * 52 + grouped['week'].astype(int) - 1

In [136]:
le_store = LabelEncoder()
le_dept = LabelEncoder()

grouped['Store'] = le_store.fit_transform(grouped['Store'])
grouped['Dept'] = le_dept.fit_transform(grouped['Dept'])

In [137]:
min_num_of_week = 0
max_num_of_week = 52*3
grouped['num_of_week'] = (grouped['num_of_week'] - 0) / max_num_of_week

In [138]:
grouped.head()

,Store,Dept,year,week,Weekly_Sales,num_of_week
0,0,0,2010,1,4565.75,0.000000
1,0,0,2010,2,4662.88,0.006410
2,0,0,2010,3,4760.60,0.012821
3,0,0,2010,4,4131.21,0.019231
4,0,0,2010,5,3973.98,0.025641


In [161]:
X_features = grouped[['Store', 'Dept', 'num_of_week']].values
y = grouped['Weekly_Sales'].values

In [163]:
y_min = y.min()
y_max = y.max()
y = (y - y_min) / (y_max - y_min)

In [164]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(256, activation='relu', input_shape=(3,)),
    tf.keras.layers.Dense(1024, activation='relu'),
    tf.keras.layers.Dense(1)
])
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
X_train, X_test, y_train, y_test = train_test_split(X_features, y, test_size=0.2, random_state=0)

In [165]:
model.fit(X_train, y_train, epochs=10, batch_size=256, validation_data=(X_test, y_test))

Epoch 1/10


163/163 [==============================] - 1s 4ms/step - loss: 0.4905 - mae: 0.2559 - val_loss: 0.0021 - val_mae: 0.0303
Epoch 2/10
163/163 [==============================] - 1s 3ms/step - loss: 0.0021 - mae: 0.0290 - val_loss: 0.0021 - val_mae: 0.0328
Epoch 3/10
163/163 [==============================] - 1s 3ms/step - loss: 0.0021 - mae: 0.0293 - val_loss: 0.0021 - val_mae: 0.0268
Epoch 4/10
163/163 [==============================] - 1s 3ms/step - loss: 0.0021 - mae: 0.0297 - val_loss: 0.0020 - val_mae: 0.0294
Epoch 5/10
163/163 [==============================] - 1s 3ms/step - loss: 0.0021 - mae: 0.0296 - val_loss: 0.0021 - val_mae: 0.0262
Epoch 6/10
163/163 [==============================] - 1s 3ms/step - loss: 0.0021 - mae: 0.0296 - val_loss: 0.0020 - val_mae: 0.0304
Epoch 7/10
163/163 [==============================] - 1s 3ms/step - loss: 0.0020 - mae: 0.0294 - val_loss: 0.0022 - val_mae: 0.0365
Epoch 8/10
163/163 [==============================] - 1s 3ms/step - loss: 0.0023 - mae:

In [169]:
model.save('../../resources/model/tpcxai_sf1/final/tf/usecase3.h5', include_optimizer=False)
model_weights = model.get_weights()
with h5py.File('../../resources/model/tpcxai_sf1/final/velox/usecase3_ffnn_weight.h5', 'w') as f:
    f.create_dataset('w1', data=model_weights[0])
    f.create_dataset('b1', data=model_weights[1])
    f.create_dataset('w2', data=model_weights[2])
    f.create_dataset('b2', data=model_weights[3])
    f.create_dataset('w3', data=model_weights[4])
    f.create_dataset('b3', data=model_weights[5])

In [195]:
with open('../../resources/model/tpcxai_sf1/final/tf/usecase3_le_store.pkl', 'wb') as f:
    pickle.dump(le_store, f)
with open('../../resources/model/tpcxai_sf1/final/tf/usecase3_le_dept.pkl', 'wb') as f:
    pickle.dump(le_dept, f)

In [186]:
# Use case 10, serving query
query_to_fetch_serving_data = """
select store, department, num_of_week from tpcxai_store_dept_serving
"""

In [187]:
df_serve = utils.fetch_data_from_postgres_via_connectorx(query_to_fetch_serving_data)

In [188]:
# df_serve = df_serve[df_serve['department'] != "ACCESSORIES"][df_serve['department'] != "PRE PACKED DELI"]

In [196]:
le_store = pickle.load(open('../../resources/model/tpcxai_sf1/final/tf/usecase3_le_store.pkl', 'rb'))
le_dept = pickle.load(open('../../resources/model/tpcxai_sf1/final/tf/usecase3_le_dept.pkl', 'rb'))

In [189]:
df_serve['store'] = le_store.transform(df_serve['store'].values)
df_serve['department'] = le_dept.transform(df_serve['department'].values)
df_serve['num_of_week'] = (df_serve['num_of_week'] - 0) / max_num_of_week

In [192]:
X_serve = df_serve[['store', 'department', 'num_of_week']].values.astype(float)
y_pred = model.predict(X_serve)

In [194]:
y_pred = y_pred * (y_max - y_min) + y_min